# Agentic Workflow with Tool Logging

This notebook demonstrates how to add comprehensive logging to monitor tools being used in an agentic workflow.

## Overview
- Setup logging infrastructure
- Create example tools
- Build an agent that uses these tools
- Monitor and log all tool invocations

## 1. Install Required Dependencies

In [ ]:
# Install required packages
!pip install anthropic python-dotenv -q

## 2. Setup Logging Infrastructure

In [ ]:
import logging
import json
from datetime import datetime
from typing import Any, Dict, List, Callable
import functools

# Configure logging with multiple handlers
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  # Console output
        logging.FileHandler('agent_workflow.log')  # File output
    ]
)

# Create specialized loggers
agent_logger = logging.getLogger('Agent')
tool_logger = logging.getLogger('ToolExecution')
performance_logger = logging.getLogger('Performance')

## 3. Create Tool Logging Decorator

In [ ]:
import time

class ToolLogger:
    """Logger for tracking tool usage in agentic workflows"""
    
    def __init__(self):
        self.tool_usage_stats = {}
        self.execution_history = []
    
    def log_tool_call(self, func: Callable) -> Callable:
        """Decorator to log tool invocations with detailed information"""
        
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            tool_name = func.__name__
            start_time = time.time()
            
            # Log tool invocation
            tool_logger.info(f"🔧 TOOL CALLED: {tool_name}")
            tool_logger.info(f"   Arguments: {args}")
            tool_logger.info(f"   Kwargs: {kwargs}")
            
            try:
                # Execute the tool
                result = func(*args, **kwargs)
                
                # Calculate execution time
                execution_time = time.time() - start_time
                
                # Log success
                tool_logger.info(f"✅ TOOL SUCCESS: {tool_name}")
                tool_logger.info(f"   Execution Time: {execution_time:.4f}s")
                tool_logger.info(f"   Result: {str(result)[:200]}...")  # Truncate long results
                
                # Update statistics
                self._update_stats(tool_name, execution_time, success=True)
                
                # Record execution
                self._record_execution(
                    tool_name=tool_name,
                    args=args,
                    kwargs=kwargs,
                    result=result,
                    execution_time=execution_time,
                    success=True
                )
                
                return result
                
            except Exception as e:
                execution_time = time.time() - start_time
                
                # Log error
                tool_logger.error(f"❌ TOOL FAILED: {tool_name}")
                tool_logger.error(f"   Error: {str(e)}")
                tool_logger.error(f"   Execution Time: {execution_time:.4f}s")
                
                # Update statistics
                self._update_stats(tool_name, execution_time, success=False)
                
                # Record execution
                self._record_execution(
                    tool_name=tool_name,
                    args=args,
                    kwargs=kwargs,
                    result=None,
                    execution_time=execution_time,
                    success=False,
                    error=str(e)
                )
                
                raise
        
        return wrapper
    
    def _update_stats(self, tool_name: str, execution_time: float, success: bool):
        """Update tool usage statistics"""
        if tool_name not in self.tool_usage_stats:
            self.tool_usage_stats[tool_name] = {
                'total_calls': 0,
                'successful_calls': 0,
                'failed_calls': 0,
                'total_time': 0.0,
                'avg_time': 0.0
            }
        
        stats = self.tool_usage_stats[tool_name]
        stats['total_calls'] += 1
        stats['total_time'] += execution_time
        stats['avg_time'] = stats['total_time'] / stats['total_calls']
        
        if success:
            stats['successful_calls'] += 1
        else:
            stats['failed_calls'] += 1
    
    def _record_execution(self, tool_name: str, args: tuple, kwargs: dict, 
                         result: Any, execution_time: float, success: bool, 
                         error: str = None):
        """Record individual tool execution"""
        execution_record = {
            'timestamp': datetime.now().isoformat(),
            'tool_name': tool_name,
            'args': str(args),
            'kwargs': str(kwargs),
            'result': str(result)[:500] if result else None,
            'execution_time': execution_time,
            'success': success,
            'error': error
        }
        self.execution_history.append(execution_record)
    
    def get_stats(self) -> Dict:
        """Get tool usage statistics"""
        return self.tool_usage_stats
    
    def get_history(self) -> List[Dict]:
        """Get execution history"""
        return self.execution_history
    
    def print_summary(self):
        """Print a summary of tool usage"""
        print("\n" + "="*60)
        print("TOOL USAGE SUMMARY")
        print("="*60)
        
        for tool_name, stats in self.tool_usage_stats.items():
            print(f"\n📊 {tool_name}:")
            print(f"   Total Calls: {stats['total_calls']}")
            print(f"   Successful: {stats['successful_calls']}")
            print(f"   Failed: {stats['failed_calls']}")
            print(f"   Avg Execution Time: {stats['avg_time']:.4f}s")
            print(f"   Total Time: {stats['total_time']:.4f}s")
        
        print("\n" + "="*60)
        print(f"Total Tool Invocations: {len(self.execution_history)}")
        print("="*60 + "\n")

# Initialize global tool logger
tool_logger_instance = ToolLogger()

## 4. Define Example Tools

In [ ]:
@tool_logger_instance.log_tool_call
def calculator(operation: str, a: float, b: float) -> float:
    """Perform basic arithmetic operations"""
    operations = {
        'add': lambda x, y: x + y,
        'subtract': lambda x, y: x - y,
        'multiply': lambda x, y: x * y,
        'divide': lambda x, y: x / y if y != 0 else None
    }
    
    if operation not in operations:
        raise ValueError(f"Unknown operation: {operation}")
    
    result = operations[operation](a, b)
    return result


@tool_logger_instance.log_tool_call
def web_search(query: str) -> Dict:
    """Simulate web search (mock implementation)"""
    # Simulate network delay
    time.sleep(0.5)
    
    return {
        'query': query,
        'results': [
            {'title': f'Result 1 for {query}', 'url': 'https://example.com/1'},
            {'title': f'Result 2 for {query}', 'url': 'https://example.com/2'},
        ],
        'count': 2
    }


@tool_logger_instance.log_tool_call
def file_reader(filename: str) -> str:
    """Read a file (mock implementation)"""
    # Simulate file reading
    mock_files = {
        'data.txt': 'This is sample data from the file.',
        'config.json': '{"setting1": true, "setting2": "value"}'
    }
    
    if filename not in mock_files:
        raise FileNotFoundError(f"File not found: {filename}")
    
    return mock_files[filename]


@tool_logger_instance.log_tool_call
def database_query(table: str, condition: str = None) -> List[Dict]:
    """Query database (mock implementation)"""
    # Simulate database query
    time.sleep(0.3)
    
    mock_data = {
        'users': [
            {'id': 1, 'name': 'Alice', 'age': 30},
            {'id': 2, 'name': 'Bob', 'age': 25},
        ],
        'products': [
            {'id': 1, 'name': 'Widget', 'price': 19.99},
            {'id': 2, 'name': 'Gadget', 'price': 29.99},
        ]
    }
    
    if table not in mock_data:
        raise ValueError(f"Table not found: {table}")
    
    return mock_data[table]


@tool_logger_instance.log_tool_call
def send_email(to: str, subject: str, body: str) -> bool:
    """Send email (mock implementation)"""
    # Simulate email sending
    time.sleep(0.2)
    
    print(f"📧 Email sent to {to}")
    print(f"   Subject: {subject}")
    print(f"   Body: {body[:50]}...")
    
    return True


# Tool registry for the agent
AVAILABLE_TOOLS = {
    'calculator': calculator,
    'web_search': web_search,
    'file_reader': file_reader,
    'database_query': database_query,
    'send_email': send_email
}

print("✅ Tools defined and registered with logging")

## 5. Create Simple Agent

In [ ]:
class SimpleAgent:
    """A simple agent that can use tools"""
    
    def __init__(self, tools: Dict[str, Callable]):
        self.tools = tools
        self.logger = logging.getLogger('Agent')
    
    def execute_task(self, task_description: str, tool_calls: List[Dict]):
        """Execute a task using specified tool calls"""
        self.logger.info(f"🤖 Agent starting task: {task_description}")
        results = []
        
        for i, tool_call in enumerate(tool_calls, 1):
            tool_name = tool_call['tool']
            args = tool_call.get('args', [])
            kwargs = tool_call.get('kwargs', {})
            
            self.logger.info(f"\n📍 Step {i}/{len(tool_calls)}: Using tool '{tool_name}'")
            
            if tool_name not in self.tools:
                self.logger.error(f"Tool not found: {tool_name}")
                continue
            
            try:
                tool = self.tools[tool_name]
                result = tool(*args, **kwargs)
                results.append({
                    'step': i,
                    'tool': tool_name,
                    'result': result,
                    'success': True
                })
            except Exception as e:
                self.logger.error(f"Tool execution failed: {e}")
                results.append({
                    'step': i,
                    'tool': tool_name,
                    'error': str(e),
                    'success': False
                })
        
        self.logger.info(f"\n🎯 Task completed: {task_description}")
        return results

# Initialize agent
agent = SimpleAgent(AVAILABLE_TOOLS)
print("✅ Agent initialized")

## 6. Run Example Workflow

In [ ]:
# Example 1: Simple calculation task
print("\n" + "="*60)
print("EXAMPLE 1: Simple Calculation")
print("="*60)

task1 = [
    {'tool': 'calculator', 'kwargs': {'operation': 'add', 'a': 10, 'b': 5}},
    {'tool': 'calculator', 'kwargs': {'operation': 'multiply', 'a': 3, 'b': 7}},
]

results1 = agent.execute_task("Perform basic calculations", task1)

In [ ]:
# Example 2: Data retrieval and search
print("\n" + "="*60)
print("EXAMPLE 2: Data Retrieval")
print("="*60)

task2 = [
    {'tool': 'file_reader', 'kwargs': {'filename': 'data.txt'}},
    {'tool': 'database_query', 'kwargs': {'table': 'users'}},
    {'tool': 'web_search', 'kwargs': {'query': 'agentic workflows'}},
]

results2 = agent.execute_task("Retrieve data from multiple sources", task2)

In [ ]:
# Example 3: Complex workflow with error handling
print("\n" + "="*60)
print("EXAMPLE 3: Complex Workflow with Error")
print("="*60)

task3 = [
    {'tool': 'database_query', 'kwargs': {'table': 'products'}},
    {'tool': 'calculator', 'kwargs': {'operation': 'divide', 'a': 100, 'b': 0}},  # This will fail
    {'tool': 'file_reader', 'kwargs': {'filename': 'nonexistent.txt'}},  # This will also fail
    {'tool': 'send_email', 'kwargs': {'to': 'user@example.com', 'subject': 'Report', 'body': 'Task completed'}},
]

results3 = agent.execute_task("Complex workflow with error handling", task3)

## 7. Analyze Tool Usage

In [ ]:
# Print tool usage summary
tool_logger_instance.print_summary()

In [ ]:
# Get detailed statistics
stats = tool_logger_instance.get_stats()
print("\n📈 Detailed Statistics:\n")
print(json.dumps(stats, indent=2))

In [ ]:
# View execution history
history = tool_logger_instance.get_history()
print(f"\n📜 Execution History ({len(history)} entries):\n")
for i, entry in enumerate(history, 1):
    status = "✅" if entry['success'] else "❌"
    print(f"{i}. {status} {entry['tool_name']} - {entry['execution_time']:.4f}s")
    if not entry['success']:
        print(f"   Error: {entry['error']}")

## 8. Visualize Tool Usage

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create visualizations
stats = tool_logger_instance.get_stats()

if stats:
    # Prepare data
    tools = list(stats.keys())
    total_calls = [stats[tool]['total_calls'] for tool in tools]
    avg_times = [stats[tool]['avg_time'] for tool in tools]
    success_rates = [(stats[tool]['successful_calls'] / stats[tool]['total_calls'] * 100) 
                     if stats[tool]['total_calls'] > 0 else 0 
                     for tool in tools]
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Tool Usage Analytics', fontsize=16, fontweight='bold')
    
    # 1. Total calls per tool
    axes[0, 0].bar(tools, total_calls, color='steelblue')
    axes[0, 0].set_title('Total Calls per Tool')
    axes[0, 0].set_xlabel('Tool')
    axes[0, 0].set_ylabel('Number of Calls')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Average execution time
    axes[0, 1].bar(tools, avg_times, color='coral')
    axes[0, 1].set_title('Average Execution Time')
    axes[0, 1].set_xlabel('Tool')
    axes[0, 1].set_ylabel('Time (seconds)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Success rate
    colors = ['green' if rate == 100 else 'orange' if rate >= 50 else 'red' 
              for rate in success_rates]
    axes[1, 0].bar(tools, success_rates, color=colors)
    axes[1, 0].set_title('Success Rate (%)')
    axes[1, 0].set_xlabel('Tool')
    axes[1, 0].set_ylabel('Success Rate')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].axhline(y=100, color='g', linestyle='--', alpha=0.3)
    
    # 4. Tool usage distribution (pie chart)
    axes[1, 1].pie(total_calls, labels=tools, autopct='%1.1f%%', startangle=90)
    axes[1, 1].set_title('Tool Usage Distribution')
    
    plt.tight_layout()
    plt.show()
else:
    print("No tool usage data available yet. Run some workflows first!")

## 9. Export Logs

In [ ]:
# Export execution history to JSON
history = tool_logger_instance.get_history()
with open('tool_execution_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("✅ Execution history exported to 'tool_execution_history.json'")

# Export statistics to JSON
stats = tool_logger_instance.get_stats()
with open('tool_usage_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print("✅ Statistics exported to 'tool_usage_stats.json'")

# Create CSV report
df = pd.DataFrame([
    {
        'Tool': tool,
        'Total Calls': data['total_calls'],
        'Successful': data['successful_calls'],
        'Failed': data['failed_calls'],
        'Avg Time (s)': round(data['avg_time'], 4),
        'Total Time (s)': round(data['total_time'], 4),
        'Success Rate (%)': round((data['successful_calls'] / data['total_calls'] * 100) 
                                   if data['total_calls'] > 0 else 0, 2)
    }
    for tool, data in stats.items()
])

df.to_csv('tool_usage_report.csv', index=False)
print("✅ Report exported to 'tool_usage_report.csv'")
print("\n📊 Report Preview:")
display(df)

## 10. Advanced: Real-time Monitoring Dashboard

In [ ]:
from IPython.display import display, HTML, clear_output
import time as time_module

def create_monitoring_dashboard():
    """Create a real-time monitoring dashboard"""
    stats = tool_logger_instance.get_stats()
    history = tool_logger_instance.get_history()
    
    html = f"""
    <div style="font-family: Arial, sans-serif; background: #f5f5f5; padding: 20px; border-radius: 10px;">
        <h2 style="color: #333;">🔍 Agent Tool Monitoring Dashboard</h2>
        <p style="color: #666;">Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        
        <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin: 20px 0;">
            <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="margin: 0; color: #2196F3;">📊 Total Tools</h3>
                <p style="font-size: 32px; margin: 10px 0; font-weight: bold;">{len(stats)}</p>
            </div>
            <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="margin: 0; color: #4CAF50;">✅ Total Calls</h3>
                <p style="font-size: 32px; margin: 10px 0; font-weight: bold;">{len(history)}</p>
            </div>
            <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="margin: 0; color: #FF9800;">⚡ Avg Time</h3>
                <p style="font-size: 32px; margin: 10px 0; font-weight: bold;">
                    {sum(e['execution_time'] for e in history) / len(history):.3f}s
                </p>
            </div>
        </div>
        
        <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin: 20px 0;">
            <h3 style="color: #333;">🔧 Tool Breakdown</h3>
            <table style="width: 100%; border-collapse: collapse;">
                <thead>
                    <tr style="background: #f0f0f0;">
                        <th style="padding: 10px; text-align: left;">Tool</th>
                        <th style="padding: 10px; text-align: center;">Calls</th>
                        <th style="padding: 10px; text-align: center;">Success</th>
                        <th style="padding: 10px; text-align: center;">Failed</th>
                        <th style="padding: 10px; text-align: right;">Avg Time</th>
                    </tr>
                </thead>
                <tbody>
    """
    
    for tool, data in stats.items():
        success_rate = (data['successful_calls'] / data['total_calls'] * 100) if data['total_calls'] > 0 else 0
        color = "#4CAF50" if success_rate == 100 else "#FF9800" if success_rate >= 50 else "#F44336"
        
        html += f"""
                    <tr style="border-bottom: 1px solid #eee;">
                        <td style="padding: 10px;"><strong>{tool}</strong></td>
                        <td style="padding: 10px; text-align: center;">{data['total_calls']}</td>
                        <td style="padding: 10px; text-align: center; color: #4CAF50;">{data['successful_calls']}</td>
                        <td style="padding: 10px; text-align: center; color: #F44336;">{data['failed_calls']}</td>
                        <td style="padding: 10px; text-align: right;">{data['avg_time']:.4f}s</td>
                    </tr>
        """
    
    html += """
                </tbody>
            </table>
        </div>
        
        <div style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h3 style="color: #333;">📜 Recent Activity</h3>
            <div style="max-height: 300px; overflow-y: auto;">
    """
    
    for entry in reversed(history[-10:]):
        icon = "✅" if entry['success'] else "❌"
        color = "#4CAF50" if entry['success'] else "#F44336"
        html += f"""
                <div style="padding: 8px; border-left: 3px solid {color}; margin: 5px 0; background: #f9f9f9;">
                    <strong>{icon} {entry['tool_name']}</strong> - {entry['execution_time']:.4f}s
                    <br><small style="color: #666;">{entry['timestamp']}</small>
                </div>
        """
    
    html += """
            </div>
        </div>
    </div>
    """
    
    return HTML(html)

# Display dashboard
display(create_monitoring_dashboard())

## Summary

This notebook demonstrates:

1. **Logging Infrastructure**: Set up comprehensive logging with multiple handlers
2. **Tool Decorator**: Created a decorator to automatically log all tool invocations
3. **Statistics Tracking**: Track usage statistics including call counts, execution times, and success rates
4. **Error Handling**: Proper error logging and tracking of failed tool calls
5. **Visualization**: Generate charts and graphs to analyze tool usage patterns
6. **Export Capabilities**: Export logs and statistics to JSON and CSV formats
7. **Real-time Dashboard**: Create an HTML dashboard for monitoring tool usage

### Key Benefits:
- **Transparency**: Know exactly which tools are being called and when
- **Performance Monitoring**: Track execution times to identify bottlenecks
- **Error Analysis**: Quickly identify and debug failing tool calls
- **Usage Analytics**: Understand tool usage patterns for optimization
- **Audit Trail**: Complete history of all tool invocations for compliance
